# Inspecting the Vietnam TB run

This notebook runs the summer4 port of the `tweak_cases_calib` Vietnam model at the default parameters and checks it against the summer3 golden. It is a forward run only. There is no calibration and no likelihood.

The model has 80 compartments: seven disease states and eight ages (`0, 3, 5, 10, 15, 18, 40, 65`), with infectiousness and clinical status only on `active`. Solver time is an index, `0` through `300`. Calendar year is that index plus 1800, so the saved series runs from 1800 to 2100. The population starts at 1000 people in `mtb_naive` at each age. Later births come from the UN increment series, not from that flat start.

In VS Code, select the kernel **tb-macro-summer4 (pixi)**. That kernel is the default pixi environment (`.pixi/envs/default`), which installs summer4. If it is missing, run `pixi run install-kernel` once from the repository root. Do not use the `reference` environment here: it pins summer3 and does not install this package.

The run below copies three summer3 bugs so the lines sit on the golden. [The next notebook](02-ReleaseModel.ipynb) turns those bugs off and shows how the epidemic changes. The bugs are written up in `reference/summer3-bugs.md`.


## Run the default parameters

`BASE_PARAMS` is the inspect-notebook default. Active case finding coverage is 0, so the `acf` flow is identically zero. Infection is seeded by a triangular pulse centred on 1830. Passive detection stays at zero until 1957, then follows the cosine scale-up through 1986, 2007 and 2014, with a dip in 2021.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from tb_macro.constants import PARITY_ATOL, PARITY_RTOL
from tb_macro.model import calendar_years, run
from tb_macro.outputs import age_incidence, age_prevalence, compartments_by_label, disease_state_totals
from tb_macro.parameters import BASE_PARAMS

result = run(BASE_PARAMS)
years = np.asarray(calendar_years())
totals = {state: np.asarray(values) for state, values in disease_state_totals(result).items()}

root = Path.cwd()
if not (root / "tests" / "golden" / "base").is_dir():
    root = root.parent
golden_dir = root / "tests" / "golden" / "base"
compartments = pd.read_parquet(golden_dir / "compartments.parquet")
flows = pd.read_parquet(golden_dir / "flows.parquet")


## Disease-state totals

Each line sums one disease state over age. For `active`, that sum also folds in infectiousness and clinical status. The seed in the 1830s is what first moves people out of `mtb_naive`. Treatment stays empty until detection begins in 1957, then tracks clinical disease through the scale-up.


In [2]:
figure = go.Figure()
for state, values in totals.items():
    figure.add_trace(go.Scatter(x=years, y=values, name=state, mode="lines"))
figure.update_layout(
    title="Disease-state totals at the default parameters",
    xaxis_title="Year",
    yaxis_title="People",
    template="plotly_white",
    legend_title="Disease state",
    height=520,
)
figure.show()


## On top of the summer3 golden

The golden is the same construction as `notebooks/03-InspectRun.ipynb` on `tweak_cases_calib`: Vietnam demography, a flat start of 1000 per age, and the summer3 solver tightened to `rtol = atol = 1e-9` because summer4 cannot set `dtmax`. The port is compared at `rtol = 1e-5` with an absolute floor of `1e-8`. Occupied compartments agree to a few parts per million. The floor sits above the pre-seed roundoff, which is about `1e-9` and is not an epidemiological difference.

Prevalence here is `active` plus `treatment`. Incidence is the progression flow, summed over age. The golden and the port are drawn together. If the port has matched, each pair is one line.


In [3]:
def _state_total(frame: pd.DataFrame, state: str) -> np.ndarray:
    columns = [column for column in frame.columns if column == state or column.startswith(state + "_")]
    return frame[columns].sum(axis=1).to_numpy()


golden_prevalence = _state_total(compartments, "active") + _state_total(compartments, "treatment")
port_prevalence = totals["active"] + totals["treatment"]
golden_incidence = flows["progression"].to_numpy()
port_incidence = sum(np.asarray(values) for values in age_incidence(result).values())

compare = go.Figure()
compare.add_trace(go.Scatter(x=years, y=golden_prevalence, name="Prevalence, summer3", mode="lines"))
compare.add_trace(
    go.Scatter(x=years, y=port_prevalence, name="Prevalence, summer4", mode="lines", line={"dash": "dash"})
)
compare.add_trace(go.Scatter(x=years, y=golden_incidence, name="Incidence, summer3", mode="lines"))
compare.add_trace(
    go.Scatter(x=years, y=port_incidence, name="Incidence, summer4", mode="lines", line={"dash": "dash"})
)
compare.update_layout(
    title="Prevalence and incidence against the summer3 golden",
    xaxis_title="Year",
    yaxis_title="People, or people per year",
    template="plotly_white",
    height=520,
)
compare.show()

got = compartments_by_label(result)
assert set(got) == set(compartments.columns) - {"time"}
for column, values in got.items():
    np.testing.assert_allclose(
        np.asarray(values),
        compartments[column].to_numpy(),
        rtol=PARITY_RTOL,
        atol=PARITY_ATOL,
    )
np.testing.assert_allclose(port_incidence, golden_incidence, rtol=PARITY_RTOL, atol=PARITY_ATOL)
print(f"base golden matched at rtol={PARITY_RTOL}, atol={PARITY_ATOL}")


base golden matched at rtol=1e-05, atol=1e-08
